# BERT Model Video Refrence
[Text Classification using Transformers | BERT | Custom Dataset | with code](https://www.youtube.com/watch?v=TmT-sKxovb0&t=747s)

# BERT Model Website Refrences
[BERT Model NLP](https://www.geeksforgeeks.org/nlp/explanation-of-bert-model-nlp/)

[AutoTokenizer HuggingFace Documentation](https://huggingface.co/docs/transformers/v4.56.0/en/model_doc/auto#transformers.AutoTokenizer)

[AutoModel HuggingFace Documentation](https://huggingface.co/docs/transformers/v4.56.0/en/model_doc/auto#transformers.AutoModel)

In [ ]:
#Importing the required libraries needed for the Phising Email Dectector
from google.colab import drive #Calling the drive within Google Drive which the CSV files are stored
import os
import pandas as pd            #Used for reading the Phising Email Dataset

In [ ]:
#Preprocessed function created which handles the preprocessing of the Phising Email CSV Dataset
#Parameter: Takes in the filepath of where the CSV Dataset is stored

file_path = ('/content/drive/MyDrive/Phising_Emails_CSVs')
def DataSet(file_path) :

  #Looping through the directory to acquire the file Datasets which have .CSV extensions
  for filename in os.listdir(file_path) :
    #If the file ends with .CSV
    if filename.endswith(".csv") :
      #Find the files within the file path that contain the .CSV extension
      csv_file_path = os.path.join(file_path, filename)
      #Read the Dataset using pandas read command
      df = pd.read_csv(csv_file_path, encoding='mac_roman')

      #Preforming Binary Classification by labeling the data in the dataset as either 0 for legitamite or 1 for phising
      df['Label'] = df['Email Type'].apply(lambda x: 1 if x == "Phishing Email" else 0)

      #Filling in empty email text cells within the CSV file to ensure they are not NaN
      df['Email Text'] = df['Email Text'].fillna(' ')

      return df
#Calling the function
df = DataSet(file_path)
#Printing out the first 5 rows of the dataset to ensure labeling has been applied
df.head()

,Unnamed: 0,Email Text,Email Type,Label
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,2,re : equistar deal tickets are you still avail...,Safe Email,0
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email,1


In [ ]:
#Importing libraries to utilize the default BERT model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
#Importing the Dataset from the dataset library
from datasets import Dataset
#Importing the test_train_split to prefrom the 80/20 train test split on the BERT model
from sklearn.model_selection import train_test_split

In [ ]:
#Intializng the tokenizer which utilzing the pretrained BERT model as a parameter
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
#Created a processing data function which takes in the text and labels from the dataset by each row
def processing_data(row) :
  #Going through each row of the email text
  text = row['Email Text']
  text = str(text)
  # Splitting the email text into words
  words = text.split()
  # Join the words back into a single string
  processed_text = ' '.join(words)


  enconding_text = tokenizer(processed_text, max_length=256, truncation=True, padding='max_length')

  enconding_text['label'] = row['Label'] # Correctly access the 'Label' column from the row
  enconding_text['text'] = processed_text # Store the processed text

  return enconding_text

In [ ]:
#Preprocessing the data and storing it within a list
processed_data = []

for i in range(len(df)) :
  processed_data.append(processing_data(df.iloc[i]))


In [ ]:
from sklearn.model_selection import train_test_split

new_df = pd.DataFrame(processed_data)

# Split the data into training and testing sets
train_df, test_df = train_test_split(new_df, test_size=0.2, random_state=2022)

In [ ]:
import pyarrow as pa
from datasets import Dataset

#Creating a train and test Dataset which would be accepted by the BERT model
train_hg = Dataset(pa.Table.from_pandas(train_df))
test_hg = Dataset(pa.Table.from_pandas(test_df))

In [ ]:
print(type(train_hg))

<class 'datasets.arrow_dataset.Dataset'>


In [ ]:
# Creating and initializaing the BERT model and having it take in the number of labels
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments, Trainer
#Intailizaing the training arguments of the BERT model
training_args = TrainingArguments(output_dir="/content/drive/MyDrive/result", eval_strategy="epoch", report_to="none")
#
trainer = Trainer(model=model, args=training_args, train_dataset=train_hg, eval_dataset=test_hg, tokenizer=tokenizer)
#Training the BERT model on the Phishing email Dataset
trainer.train()

/tmp/ipython-input-3859332180.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=train_hg, eval_dataset=test_hg, tokenizer=tokenizer)


Epoch,Training Loss,Validation Loss
1,0.120500,0.121995
2,0.057600,0.079803
3,0.029200,0.085725


TrainOutput(global_step=5595, training_loss=0.07838668471686643, metrics={'train_runtime': 3143.5586, 'train_samples_per_second': 14.239, 'train_steps_per_second': 1.78, 'total_flos': 5888425418956800.0, 'train_loss': 0.07838668471686643, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.08572500944137573,
 'eval_runtime': 54.5201,
 'eval_samples_per_second': 68.415,
 'eval_steps_per_second': 8.566,
 'epoch': 3.0}

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import torch
from transformers import Trainer, TrainingArguments

# Having the BERT model make the predictions from the untrained portion of the Dataset
predictions = trainer.predict(test_hg)

# The predictions are in the 'predictions' attribute of the output
# The true labels are in the 'label_ids' attribute
y_true = predictions.label_ids
y_pred_logits = predictions.predictions

# Convert logits to predicted class labels
y_pred = np.argmax(y_pred_logits, axis=1)

# Calculating and Printing out the Accuracy Score of the BERT Model
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy Score: {accuracy * 100:.2f}%")

# Printing Classification Report to the console
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Legitimate Email", "Phishing Email"]))

# Printing Confusion Matrix to the consle
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy Score: 97.88%

Classification Report:
                  precision    recall  f1-score   support

Legitimate Email       0.99      0.98      0.98      2202
  Phishing Email       0.97      0.98      0.97      1528

        accuracy                           0.98      3730
       macro avg       0.98      0.98      0.98      3730
    weighted avg       0.98      0.98      0.98      3730


Confusion Matrix:
[[2148   54]
 [  25 1503]]


In [ ]:
#Saving the BERT model after training and evaluation have been complete
model = model.save_pretrained("/content/drive/MyDrive/bert_phishing_model")

In [ ]:
#Saving the tokenizer after training and evaluation have been complete
tokenizer = tokenizer.save_pretrained("/content/drive/MyDrive/bert_phishing_model")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
#Loading in the trained BERT model and tokenizer to test through developed Python inference code
new_model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/bert_phishing_model")
new_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/bert_phishing_model")

In [ ]:

def bertPredict(email_text):
    #Tokenizes the email text being taken in from the prompt
    inputs = new_tokenizer(
        email_text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    #Handles making the prediction of the email text
    with torch.no_grad():
        outputs = new_model(**inputs)
        logits = outputs.logits

    # Convert logits to probabilities
    probabilities = torch.softmax(logits, dim=1).squeeze().tolist()

    # Determine predicted label to be either Phishing or Legitimate
    predicted_class = torch.argmax(logits, dim=1).item()
    predicted_label = "Phishing Email" if predicted_class == 1 else "Legitimate Email"

    # Extracts the probabilities and prints them out to the console
    legitimate_conf = round(probabilities[0] * 100, 2)
    phishing_conf = round(probabilities[1] * 100, 2)
    probs_dict = {
        "Legitimate Email": f"{legitimate_conf}%",
        "Phishing Email": f"{phishing_conf}%"
    }
    # Returning the values of the label and proabilities
    return predicted_label, probs_dict

if __name__ == "__main__":
    #Taking in the email text from the user
    email_input = input("Enter an email text you would like to classify:\n")
    #Calling the bertPredict function to process the email text
    label, probs = bertPredict(email_input)
    #Printing the label and proabilities to the console
    print(f"Predicted Label: {label}")
    print("Probabilities:")
    for k, v in probs.items():
        print(f"{k}: {v}")


Enter an email text you would like to classify:
Hello,  This is the IRS contacting you because you have a lock on your account. Please send an amount of $300 dollars to reinstate your account, otherwise the authorities will show up to your door to which you will serve 15 years in jail if the amount has not been paid.  Thanks
Predicted Label: Phishing Email
Probabilities:
Legitimate Email: 0.01%
Phishing Email: 99.99%


# Refrences
[Pytorch Softmax Function Documentaion](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html)

[Pytorch Squeeze Function Documentation](https://docs.pytorch.org/docs/stable/generated/torch.squeeze.html)

[Pytorch Squeeze StackOverflow](https://stackoverflow.com/questions/61598771/squeeze-vs-unsqueeze-in-pytorch)

[Pytorch Argmax Function Documentation](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

[Pytorch No_Grad Function Documentation](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)
